In [3]:
import numpy as np
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [4]:
from torch.utils.data import Subset, DataLoader
from sqrlearning.server import FedAvgServer, KrumServer, TrimmedMeanServer
from sqrlearning.client import DPClient
from models.nets import MnistCNN
from dataset.noniid import mnist_data
from fedlearning import *

local_epochs = 5
num_clients = 5
batch_size = 64

train_data, test_data, clients_dataset = mnist_data(num_clients, alpha=0.5, seed=42)

def class_counts(subset, num_classes=10):
    indices = list(subset.indices)
    base = subset.dataset
    while isinstance(base, Subset):
        indices = [base.indices[i] for i in indices]
        base = base.dataset

    targets = getattr(base, "targets", None)
    labels = ([int(targets[i]) for i in indices] if targets is not None
              else [int(base[i][1]) for i in indices])
    
    return np.bincount(labels, minlength=num_classes)

for i, subset in enumerate(clients_dataset):
    counts = class_counts(subset)
    total = counts.sum()
    print(f"Client-{i}: total={total}")
    line_1 = ' '.join([f"\tclass-{i} {100 * c / total:.1f}%({c})" for i, c in enumerate(counts[:5])])
    print(" " + line_1)
    line_2 = ' '.join([f"\tclass-{i+5} {100 * c / total:.1f}%({c})" for i, c in enumerate(counts[5:])])
    print(" " + line_2)

Client-0: total=7638
 	class-0 3.6%(278) 	class-1 7.6%(577) 	class-2 2.1%(163) 	class-3 0.0%(3) 	class-4 48.9%(3733)
 	class-5 19.9%(1521) 	class-6 0.4%(28) 	class-7 6.2%(477) 	class-8 0.2%(14) 	class-9 11.1%(844)
Client-1: total=22223
 	class-0 4.3%(953) 	class-1 25.0%(5559) 	class-2 19.7%(4371) 	class-3 19.8%(4397) 	class-4 2.6%(567)
 	class-5 10.9%(2423) 	class-6 16.5%(3677) 	class-7 0.7%(159) 	class-8 0.2%(39) 	class-9 0.4%(78)
Client-2: total=8960
 	class-0 7.4%(665) 	class-1 5.5%(491) 	class-2 0.1%(13) 	class-3 2.0%(181) 	class-4 13.1%(1178)
 	class-5 0.2%(16) 	class-6 1.6%(142) 	class-7 42.5%(3804) 	class-8 5.3%(476) 	class-9 22.3%(1994)
Client-3: total=14313
 	class-0 23.0%(3291) 	class-1 0.8%(115) 	class-2 2.1%(303) 	class-3 1.1%(161) 	class-4 0.6%(90)
 	class-5 10.0%(1430) 	class-6 11.6%(1663) 	class-7 12.5%(1787) 	class-8 29.1%(4171) 	class-9 9.1%(1302)
Client-4: total=6866
 	class-0 10.7%(736) 	class-1 0.0%(0) 	class-2 16.1%(1108) 	class-3 20.2%(1389) 	class-4 4.0%(274)
 	c

In [ ]:
def test_dpclient(rounds, batch_size, target_epsilon, **kwargs):
    
    server = FedAvgServer(MnistCNN(), device=device)
        # server = TrimmedMeanServer(CNNMnist(num_channels, num_classes), num_attackers=1, device=device)

    clients = [DPClient(f"Client-{i}",
                    clients_dataset[i],
                    MnistCNN,
                    batch_size=batch_size,
                    num_epochs=local_epochs,
                    total_rounds=rounds,
                    target_epsilon=target_epsilon,
                    device=device, **kwargs)
                    for i in range(num_clients)
               ]
    for c in clients:
        print(f"{c.name}: {c.data_size} samples, {c.noise_info}")

    for round in range(1, rounds + 1):
        print(f"Round {round:2d}:")
        train(server, clients)

        acc_train, loss_avg = evaluate(server.net, train_data, device)
        # acc_attacher, _ = evaluate(clients[0].net, clients[0].train_data, device)

        print(f"Round {round:2d}: Training acc: {100*acc_train:.2f}%, global loss {loss_avg:.3f}")
        # print(f"Round {round:2d}: Attacker training acc: {100*acc_attacher:.2f}%")
        # asr_global = evaluate_asr(server.net, dataset_test, trigger_fn, target_label, device)
        # asr_local = evaluate_asr(clients[0].net, dataset_test, trigger_fn, target_label, device)
        # print(f"Round {round:2d}: global ASR={100*asr_global:.2f}%, attacker-local ASR={100*asr_local:.2f}%")

    acc_test, _ = evaluate(server.net, test_data, device)
    print(f"Testing accuracy: {100*acc_test:.2f}%")

    for c in clients:
        c.cleanup()

In [ ]:
test_dpclient(rounds = 15, batch_size = 32, target_epsilon = 50, target_delta = 1e-4)

Client-0: 7638 samples, sigma=0.4045677185058594
Client-1: 22223 samples, sigma=0.36652565002441406
Client-2: 8960 samples, sigma=0.39821624755859375
Client-3: 14313 samples, sigma=0.3809165954589844
Client-4: 6866 samples, sigma=0.4090118408203125
[1 0 2 3 4] in this round
Client-1: ε=13.133 (target=50), δ=0.0001
Client-0: ε=12.649 (target=50), δ=0.0001
Client-2: ε=12.725 (target=50), δ=0.0001
Client-3: ε=12.940 (target=50), δ=0.0001
Client-4: ε=12.595 (target=50), δ=0.0001
Round  1: Training acc: 46.74%, global loss 1.771
[0 4 2 3 1] in this round
Client-0: ε=16.881 (target=50), δ=0.0001
Client-4: ε=16.829 (target=50), δ=0.0001
Client-2: ε=16.955 (target=50), δ=0.0001
Client-3: ε=17.165 (target=50), δ=0.0001
Client-1: ε=17.352 (target=50), δ=0.0001
Round  2: Training acc: 57.50%, global loss 1.147
[2 0 3 1 4] in this round
Client-2: ε=20.454 (target=50), δ=0.0001
Client-0: ε=20.383 (target=50), δ=0.0001
Client-3: ε=20.652 (target=50), δ=0.0001
Client-1: ε=20.829 (target=50), δ=0.0001

In [ ]:
test_dpclient(rounds = 15, batch_size = 64, target_epsilon = 50, target_delta = 1e-4)

Client-0: 7638 samples, sigma=0.4358100891113281
Client-1: 22223 samples, sigma=0.3899574279785156
Client-2: 8960 samples, sigma=0.4282379150390625
Client-3: 14313 samples, sigma=0.4072380065917969
Client-4: 6866 samples, sigma=0.4411888122558594
[2 3 0 1 4] in this round
Client-2: ε=12.388 (target=50), δ=0.0001
Client-3: ε=12.619 (target=50), δ=0.0001
Client-0: ε=12.310 (target=50), δ=0.0001
Client-1: ε=12.824 (target=50), δ=0.0001
Client-4: ε=12.256 (target=50), δ=0.0001
Round  1: Training acc: 22.09%, global loss 2.106
[1 3 2 4 0] in this round
Client-1: ε=17.052 (target=50), δ=0.0001
Client-3: ε=16.852 (target=50), δ=0.0001
Client-2: ε=16.628 (target=50), δ=0.0001
Client-4: ε=16.501 (target=50), δ=0.0001
Client-0: ε=16.552 (target=50), δ=0.0001
Round  2: Training acc: 51.59%, global loss 1.662
[3 0 4 1 2] in this round
Client-3: ε=20.356 (target=50), δ=0.0001
Client-0: ε=20.073 (target=50), δ=0.0001
Client-4: ε=20.024 (target=50), δ=0.0001
Client-1: ε=20.544 (target=50), δ=0.0001
C

In [ ]:
test_dpclient(rounds = 15, batch_size = 64, target_epsilon = 50, target_delta = 1e-3)

C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


Client-0: 7638 samples, sigma=0.4212379455566406
Client-1: 22223 samples, sigma=0.3781414031982422
Client-2: 8960 samples, sigma=0.41416168212890625
Client-3: 14313 samples, sigma=0.39443016052246094
Client-4: 6866 samples, sigma=0.42629241943359375
[2 0 1 3 4] in this round


c:\Users\yuloeh\Projects\Fed-Backdoor\sqrlearning\client\DPClient.py:47: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Client-2: ε=10.770 (target=50), δ=0.001
Client-0: ε=10.714 (target=50), δ=0.001
Client-1: ε=11.095 (target=50), δ=0.001
Client-3: ε=10.942 (target=50), δ=0.001
Client-4: ε=10.672 (target=50), δ=0.001
Round  1: Training acc: 31.47%, global loss 2.070
[3 1 4 2 0] in this round
Client-3: ε=15.182 (target=50), δ=0.001
Client-1: ε=15.335 (target=50), δ=0.001
Client-4: ε=14.915 (target=50), δ=0.001
Client-2: ε=15.012 (target=50), δ=0.001
Client-0: ε=14.956 (target=50), δ=0.001
Round  2: Training acc: 46.09%, global loss 1.643
[1 0 2 3 4] in this round
Client-1: ε=18.894 (target=50), δ=0.001
Client-0: ε=18.531 (target=50), δ=0.001
Client-2: ε=18.584 (target=50), δ=0.001
Client-3: ε=18.748 (target=50), δ=0.001
Client-4: ε=18.491 (target=50), δ=0.001
Round  3: Training acc: 51.88%, global loss 1.371
[2 0 1 3 4] in this round
Client-2: ε=21.814 (target=50), δ=0.001
Client-0: ε=21.765 (target=50), δ=0.001
Client-1: ε=22.105 (target=50), δ=0.001
Client-3: ε=21.968 (target=50), δ=0.001
Client-4: ε=

In [ ]:
test_dpclient(rounds = 15, batch_size = 128, target_epsilon = 50, target_delta = 1e-3)

Client-0: 7638 samples, sigma=0.457763671875
Client-1: 22223 samples, sigma=0.40470123291015625
Client-2: 8960 samples, sigma=0.44891357421875
Client-3: 14313 samples, sigma=0.4245185852050781
Client-4: 6866 samples, sigma=0.4640960693359375
[4 2 3 1 0] in this round
Client-4: ε=10.401 (target=50), δ=0.001
Client-2: ε=10.505 (target=50), δ=0.001
Client-3: ε=10.688 (target=50), δ=0.001
Client-1: ε=10.851 (target=50), δ=0.001
Client-0: ε=10.444 (target=50), δ=0.001
Round  1: Training acc: 27.06%, global loss 2.250
[4 2 1 3 0] in this round
Client-4: ε=14.647 (target=50), δ=0.001
Client-2: ε=14.749 (target=50), δ=0.001
Client-1: ε=15.092 (target=50), δ=0.001
Client-3: ε=14.930 (target=50), δ=0.001
Client-0: ε=14.689 (target=50), δ=0.001
Round  2: Training acc: 37.52%, global loss 2.090
[4 1 3 2 0] in this round
Client-4: ε=18.236 (target=50), δ=0.001
Client-1: ε=18.661 (target=50), δ=0.001
Client-3: ε=18.507 (target=50), δ=0.001
Client-2: ε=18.333 (target=50), δ=0.001
Client-0: ε=18.276 (

In [ ]:
test_dpclient(rounds = 15, batch_size = 256, target_epsilon = 50, target_delta = 1e-3)

C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the smallest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


Client-0: 13802 samples, sigma=0.4640960693359375
Client-1: 10591 samples, sigma=0.4802131652832031
Client-2: 14874 samples, sigma=0.4587554931640625
Client-3: 8293 samples, sigma=0.4971504211425781
Client-4: 12440 samples, sigma=0.47016143798828125
[1 4 0 2 3] in this round
Client-1: ε=10.298 (target=50), δ=0.001
Client-4: ε=10.361 (target=50), δ=0.001
Client-0: ε=10.401 (target=50), δ=0.001
Client-2: ε=10.437 (target=50), δ=0.001
Client-3: ε=10.196 (target=50), δ=0.001
Round  1: Training acc: 13.60%, global loss 2.286
[3 1 4 0 2] in this round
Client-3: ε=14.444 (target=50), δ=0.001
Client-1: ε=14.545 (target=50), δ=0.001
Client-4: ε=14.607 (target=50), δ=0.001
Client-0: ε=14.647 (target=50), δ=0.001
Client-2: ε=14.683 (target=50), δ=0.001
Round  2: Training acc: 25.88%, global loss 2.250
[4 2 1 0 3] in this round
Client-4: ε=18.198 (target=50), δ=0.001
Client-2: ε=18.270 (target=50), δ=0.001
Client-1: ε=18.138 (target=50), δ=0.001
Client-0: ε=18.236 (target=50), δ=0.001
Client-3: ε=

In [ ]:
test_dpclient(rounds = 15, batch_size = 128, target_epsilon = 100, target_delta = 1e-3)

Client-0: 13802 samples, sigma=0.35805702209472656
Client-1: 10591 samples, sigma=0.36696434020996094
Client-2: 14874 samples, sigma=0.3554725646972656
Client-3: 8293 samples, sigma=0.375823974609375
Client-4: 12440 samples, sigma=0.3612709045410156
[4 1 0 3 2] in this round
Client-4: ε=19.197 (target=100), δ=0.001
Client-1: ε=19.119 (target=100), δ=0.001
Client-0: ε=19.241 (target=100), δ=0.001
Client-3: ε=19.000 (target=100), δ=0.001
Client-2: ε=19.323 (target=100), δ=0.001
Round  1: Training acc: 26.23%, global loss 2.232
[3 1 0 4 2] in this round
Client-3: ε=27.302 (target=100), δ=0.001
Client-1: ε=27.423 (target=100), δ=0.001
Client-0: ε=27.547 (target=100), δ=0.001
Client-4: ε=27.502 (target=100), δ=0.001
Client-2: ε=27.654 (target=100), δ=0.001
Round  2: Training acc: 37.96%, global loss 2.118
[0 2 1 4 3] in this round
Client-0: ε=34.672 (target=100), δ=0.001
Client-2: ε=34.799 (target=100), δ=0.001
Client-1: ε=34.551 (target=100), δ=0.001
Client-4: ε=34.628 (target=100), δ=0.00

In [ ]:
test_dpclient(rounds = 15, batch_size = 256, target_epsilon = 100, target_delta = 1e-3)

Client-0: 13802 samples, sigma=0.38295745849609375
Client-1: 10591 samples, sigma=0.39325714111328125
Client-2: 14874 samples, sigma=0.3795051574707031
Client-3: 8293 samples, sigma=0.403900146484375
Client-4: 12440 samples, sigma=0.38684844970703125
[4 0 2 1 3] in this round
Client-4: ε=18.859 (target=100), δ=0.001
Client-0: ε=18.908 (target=100), δ=0.001
Client-2: ε=18.952 (target=100), δ=0.001
Client-1: ε=18.778 (target=100), δ=0.001
Client-3: ε=18.648 (target=100), δ=0.001
Round  1: Training acc: 13.41%, global loss 2.270
[1 3 2 0 4] in this round
Client-1: ε=27.077 (target=100), δ=0.001
Client-3: ε=26.945 (target=100), δ=0.001
Client-2: ε=27.253 (target=100), δ=0.001
Client-0: ε=27.208 (target=100), δ=0.001
Client-4: ε=27.158 (target=100), δ=0.001
Round  2: Training acc: 18.36%, global loss 2.233
[0 4 3 1 2] in this round
Client-0: ε=34.342 (target=100), δ=0.001
Client-4: ε=34.293 (target=100), δ=0.001
Client-3: ε=34.086 (target=100), δ=0.001
Client-1: ε=34.214 (target=100), δ=0.0

In [ ]:
test_dpclient(rounds = 30, batch_size = 128, target_epsilon = 50, target_delta = 1e-3)

Client-0: 13802 samples, sigma=0.48152923583984375
Client-1: 10591 samples, sigma=0.4994773864746094
Client-2: 14874 samples, sigma=0.47641754150390625
Client-3: 8293 samples, sigma=0.5178070068359375
Client-4: 12440 samples, sigma=0.4879570007324219
[1 3 0 2 4] in this round
Client-1: ε=6.924 (target=50), δ=0.001
Client-3: ε=6.829 (target=50), δ=0.001
Client-0: ε=7.026 (target=50), δ=0.001
Client-2: ε=7.081 (target=50), δ=0.001
Client-4: ε=6.988 (target=50), δ=0.001
Round  1: Training acc: 17.72%, global loss 2.239
[1 0 3 4 2] in this round
Client-1: ε=9.591 (target=50), δ=0.001
Client-0: ε=9.696 (target=50), δ=0.001
Client-3: ε=9.493 (target=50), δ=0.001
Client-4: ε=9.657 (target=50), δ=0.001
Client-2: ε=9.763 (target=50), δ=0.001
Round  2: Training acc: 41.72%, global loss 2.094
[4 2 1 0 3] in this round
Client-4: ε=11.885 (target=50), δ=0.001
Client-2: ε=12.001 (target=50), δ=0.001
Client-1: ε=11.819 (target=50), δ=0.001
Client-0: ε=11.924 (target=50), δ=0.001
Client-3: ε=11.722 (t

In [ ]:
test_dpclient(rounds = 30, batch_size = 256, target_epsilon = 50, target_delta = 1e-3)

Client-0: 11743 samples, sigma=0.5469894409179688
Client-1: 11411 samples, sigma=0.5489730834960938
Client-2: 11778 samples, sigma=0.5450439453125
Client-3: 14212 samples, sigma=0.5298805236816406
Client-4: 10856 samples, sigma=0.5531692504882812
[0 2 4 1 3] in this round
Client-0: ε=6.692 (target=50), δ=0.001
Client-2: ε=6.701 (target=50), δ=0.001
Client-4: ε=6.666 (target=50), δ=0.001
Client-1: ε=6.684 (target=50), δ=0.001
Client-3: ε=6.770 (target=50), δ=0.001
Round  1: Training acc: 17.26%, global loss 2.279
[4 1 2 3 0] in this round
Client-4: ε=9.327 (target=50), δ=0.001
Client-1: ε=9.346 (target=50), δ=0.001
Client-2: ε=9.363 (target=50), δ=0.001
Client-3: ε=9.434 (target=50), δ=0.001
Client-0: ε=9.354 (target=50), δ=0.001
Round  2: Training acc: 20.20%, global loss 2.233
[3 1 2 0 4] in this round
Client-3: ε=11.663 (target=50), δ=0.001
Client-1: ε=11.575 (target=50), δ=0.001
Client-2: ε=11.592 (target=50), δ=0.001
Client-0: ε=11.583 (target=50), δ=0.001
Client-4: ε=11.557 (targe

In [ ]:
test_dpclient(rounds = 30, batch_size = 128, target_epsilon = 100, target_delta = 1e-3)

C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(
C:\Users\yuloeh\AppData\Roaming\Python\Python312\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the smallest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(


Client-0: 13802 samples, sigma=0.40640830993652344
Client-1: 10591 samples, sigma=0.4185295104980469
Client-2: 14874 samples, sigma=0.4029083251953125
Client-3: 8293 samples, sigma=0.4307365417480469
Client-4: 12440 samples, sigma=0.4107666015625
[3 1 2 4 0] in this round


c:\Users\yuloeh\Projects\Fed-Backdoor\sqrlearning\client\DPClient.py:47: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Client-3: ε=12.319 (target=100), δ=0.001
Client-1: ε=12.437 (target=100), δ=0.001
Client-2: ε=12.632 (target=100), δ=0.001
Client-4: ε=12.514 (target=100), δ=0.001
Client-0: ε=12.559 (target=100), δ=0.001
Round  1: Training acc: 23.24%, global loss 2.251
[0 2 3 4 1] in this round
Client-0: ε=17.659 (target=100), δ=0.001
Client-2: ε=17.752 (target=100), δ=0.001
Client-3: ε=17.404 (target=100), δ=0.001
Client-4: ε=17.612 (target=100), δ=0.001
Client-1: ε=17.530 (target=100), δ=0.001
Round  2: Training acc: 39.48%, global loss 2.150
[3 2 0 1 4] in this round
Client-3: ε=21.719 (target=100), δ=0.001
Client-2: ε=22.087 (target=100), δ=0.001
Client-0: ε=21.977 (target=100), δ=0.001
Client-1: ε=21.846 (target=100), δ=0.001
Client-4: ε=21.930 (target=100), δ=0.001
Round  3: Training acc: 50.23%, global loss 1.992
[4 3 2 0 1] in this round
Client-4: ε=25.848 (target=100), δ=0.001
Client-3: ε=25.637 (target=100), δ=0.001
Client-2: ε=26.020 (target=100), δ=0.001
Client-0: ε=25.896 (target=100), δ